# How Markets React to Fed Rate Decisions (2025–2026)

**Question:** When the Federal Reserve announces a rate decision, how do Treasury yields, the dollar, stocks and gold react, and did the September 2026 hike, the Fed's first since 2023, play out the way finance theory predicts?

**Data (free, public):**
- **FRED** (Federal Reserve Bank of St. Louis): Fed target range, 2-year and 10-year Treasury yields, 10-year real (TIPS) yield, broad dollar index, S&P 500
- **Yahoo Finance** via `yfinance`: COMEX gold futures (FRED stopped carrying daily gold prices in 2022)
- **Federal Reserve** meeting calendar: 15 policy decisions, January 2025 to October 2026

**Method:** an event study. For each decision, measure the move from the previous day's close to the decision-day close, then compare Fed days with ordinary days.

In [ ]:
# Setup: works in Google Colab (clones the repo) or locally from the notebooks/ folder
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/alcarrui-spec/fed-decision-market-reactions.git"
if Path("src").exists():
    ROOT = Path(".").resolve()
elif Path("../src").exists():
    ROOT = Path("..").resolve()
else:
    if not Path("fed-decision-market-reactions").exists():
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    ROOT = Path("fed-decision-market-reactions").resolve()
sys.path.insert(0, str(ROOT))

try:
    import yfinance  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)

print("Project folder:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

from src.data import load_market_data
from src.fomc import DECISION_DATES, NEXT_MEETING
from src.event_study import (LABELS, decision_day_moves, fed_day_vs_normal,
                             event_window, daily_changes)

IMAGES = ROOT / "images"
IMAGES.mkdir(exist_ok=True)
pd.set_option("display.width", 140)

# Chart style: blue = cut, orange = hike, gray = hold; quiet axes
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"
DECISION_COLORS = {"Cut": BLUE, "Hike": ORANGE, "Hold": "#9a9893"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": MUTED, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
})

## 1. Load and check the data

In [ ]:
df = load_market_data(start="2024-10-01", refresh=True)
print(f"{len(df):,} trading days, {df.index.min():%b %d, %Y} to {df.index.max():%b %d, %Y}")
print("\nMissing values by series (holidays and data gaps):")
print(df.isna().sum().to_string())
df.tail()

## 2. What did the Fed decide?

Decisions are read from the Fed's own target range (FRED `DFEDTARU`), not typed by hand.

In [ ]:
moves = decision_day_moves(df, DECISION_DATES)
decisions = moves[["decision", "change_bp", "target_range"]]
print(decisions["decision"].value_counts().to_string(), "\n")
decisions

In [ ]:
# Chart 1: the Fed's target range vs. 2-year and 10-year Treasury yields
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.fill_between(df.index, df["fed_lower"], df["fed_upper"], step="post",
                color=GRID, label="Fed target range")
ax.plot(df.index, df["yield_2y"], color=BLUE, linewidth=2, label="2-year Treasury")
ax.plot(df.index, df["yield_10y"], color=ORANGE, linewidth=2, label="10-year Treasury")
changes = decisions[decisions["decision"] != "Hold"]
for d, row in changes.iterrows():
    ax.axvline(d, color=DECISION_COLORS[row["decision"]], linewidth=1, linestyle=":")
# label each run of same-direction moves once (e.g., "3 cuts, Sep–Dec 2025")
top = ax.get_ylim()[1]
runs = (changes["decision"] != changes["decision"].shift()).cumsum()
for _, grp in changes.groupby(runs):
    kind, n = grp["decision"].iloc[0], len(grp)
    when = f"{grp.index[0]:%b %Y}" if n == 1 else f"{grp.index[0]:%b}–{grp.index[-1]:%b %Y}"
    text = f"{kind}\n{when}" if n == 1 else f"{n} {kind.lower()}s\n{when}"
    ax.annotate(text, (grp.index[0], top), xytext=(4, -2), textcoords="offset points",
                va="top", fontsize=8.5, color=INK)
ax.set_ylabel("Percent")
ax.set_title("The Fed's target range vs. Treasury yields")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(frameon=False, loc="lower left", ncol=3)
fig.text(0.125, -0.03, "Source: FRED (DFEDTARL, DFEDTARU, DGS2, DGS10). Dotted lines mark rate changes.",
         color=MUTED, fontsize=8.5)
fig.savefig(IMAGES / "01_fed_vs_yields.png")
plt.show()

## 3. How markets moved on each decision day

Yields in basis points (1 bp = 0.01%); the dollar, S&P 500 and gold in percent.

In [ ]:
cols = [c for c in LABELS if c in moves]
table = moves[["decision", "target_range"] + cols].rename(columns=LABELS)
table.round(2)

In [ ]:
# Are Fed days bigger than ordinary days?
vs_normal = fed_day_vs_normal(df, DECISION_DATES)
vs_normal.columns = ["Fed days (avg size of move)", "Other days (avg size of move)", "Fed ÷ other"]
vs_normal

In [ ]:
# Average reaction by type of decision (small samples: read as descriptive, not proof)
avg = moves.groupby("decision")[cols].mean().rename(columns=LABELS).round(2)
avg.insert(0, "meetings", moves["decision"].value_counts())
avg

In [ ]:
# Chart 2: decision-day moves in the 2-year yield and gold, one bar per meeting
has_gold = "gold" in moves and moves["gold"].notna().any()
panels = [("yield_2y", "2-year Treasury yield (bp)")] + ([("gold", "Gold (%)")] if has_gold else [])
fig, axes = plt.subplots(len(panels), 1, figsize=(10, 3.2 * len(panels)), sharex=True)
axes = np.atleast_1d(axes)
x = np.arange(len(moves))
for ax, (col, title) in zip(axes, panels):
    ax.bar(x, moves[col], color=[DECISION_COLORS[d] for d in moves["decision"]],
           width=0.66, edgecolor="white", linewidth=2)
    ax.axhline(0, color=MUTED, linewidth=1)
    ax.set_title(title)
axes[-1].set_xticks(x, [d.strftime("%b\n%Y") for d in moves.index], fontsize=8.5)
axes[0].legend(handles=[Patch(color=c, label=k) for k, c in DECISION_COLORS.items()],
               frameon=False, ncol=3, loc="upper left")
fig.suptitle("Decision-day reactions, 2025–2026", x=0.125, ha="left", fontweight="bold", fontsize=13)
fig.text(0.125, -0.02, "Change from the previous day's close to the decision-day close. "
         "Source: FRED, Yahoo Finance (GC=F).", color=MUTED, fontsize=8.5)
fig.tight_layout()
fig.savefig(IMAGES / "02_decision_day_moves.png")
plt.show()

## 4. Case study: the September 16, 2026 hike

The Fed raised rates 25 bp to 3.75%–4.00%, its first hike since July 2023. How did markets move in the 20 trading days before and after?

In [ ]:
HIKE = pd.Timestamp("2026-09-16")
path = event_window(df, HIKE, days=20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for col, color in [("yield_2y", BLUE), ("yield_10y", ORANGE)]:
    ax1.plot(path.index, path[col], color=color, linewidth=2, label=LABELS[col].replace(" (bp)", ""))
ax1.set_title("Yields (bp change)")
price_cols = [("dollar_index", BLUE), ("sp500", ORANGE)] + ([("gold", INK)] if "gold" in path else [])
for col, color in price_cols:
    ax2.plot(path.index, path[col], color=color, linewidth=2, label=LABELS[col].replace(" (%)", ""))
ax2.set_title("Prices (% change)")
for ax in (ax1, ax2):
    ax.axvline(0, color=MUTED, linewidth=1, linestyle=":")
    ax.axhline(0, color=MUTED, linewidth=0.8)
    ax.set_xlabel("Trading days from the decision (0 = Sept 16)")
    ax.legend(frameon=False, fontsize=9)
fig.suptitle("Around the September 2026 hike", x=0.07, ha="left", fontweight="bold", fontsize=13)
fig.text(0.07, -0.03, "Change vs. the close on Sept 15, 2026. Source: FRED, Yahoo Finance (GC=F).",
         color=MUTED, fontsize=8.5)
fig.tight_layout()
fig.savefig(IMAGES / "03_september_2026_hike.png")
plt.show()

snap = path.loc[[-20, 0, 20]].rename(columns=LABELS).round(2)
snap.index = ["20 days before", "Decision day", "20 days after"]
snap

## 5. Gold and real interest rates

Gold pays no interest, so when real (inflation-adjusted) yields rise, holding gold costs more. My earlier gold return model project (MAS 766) found the same relationship. Does it hold day to day in 2025–2026?

In [ ]:
if has_gold:
    chg = daily_changes(df)[["real_yield_10y", "gold"]].dropna()
    r = chg["real_yield_10y"].corr(chg["gold"])
    slope, intercept = np.polyfit(chg["real_yield_10y"], chg["gold"], 1)

    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.scatter(chg["real_yield_10y"], chg["gold"], s=14, color=BLUE, alpha=0.45, edgecolor="none")
    xs = np.linspace(chg["real_yield_10y"].min(), chg["real_yield_10y"].max(), 50)
    ax.plot(xs, intercept + slope * xs, color=ORANGE, linewidth=2)
    ax.axhline(0, color=MUTED, linewidth=0.8); ax.axvline(0, color=MUTED, linewidth=0.8)
    ax.set_xlabel("Daily change in 10-year real yield (bp)")
    ax.set_ylabel("Daily gold return (%)")
    ax.set_title("When real yields rise, gold tends to fall")
    ax.text(0.98, 0.95, f"r = {r:.2f}\n{slope*10:+.2f}% gold per +10 bp", transform=ax.transAxes,
            ha="right", va="top", fontsize=10, color=INK)
    fig.text(0.125, -0.02, f"{len(chg):,} trading days, {chg.index.min():%b %Y}–{chg.index.max():%b %Y}. "
             "Source: FRED (DFII10), Yahoo Finance (GC=F).", color=MUTED, fontsize=8.5)
    fig.savefig(IMAGES / "04_gold_vs_real_yield.png")
    plt.show()
else:
    print("Gold data unavailable; skipping this section.")

## 6. Summary for the README

In [ ]:
def fmt(v, unit):
    return "n/a" if pd.isna(v) else f"{v:+.1f} {unit}"

print("DECISIONS:", decisions["decision"].value_counts().to_dict())
print("\nFED DAYS vs OTHER DAYS (average size of move):")
print(vs_normal.to_string())
print("\nAVERAGE DECISION-DAY MOVE BY DECISION TYPE:")
print(avg.to_string())
print("\nSEPT 16, 2026 HIKE, decision day:")
d0 = moves.loc[HIKE]
for col in cols:
    print(f"  {LABELS[col]:<22} {fmt(d0[col], '')}")
print("\nSEPT 2026 HIKE, 20 days before -> 20 days after (cumulative):")
print(snap.to_string())
if has_gold:
    print(f"\nGOLD vs REAL YIELD: r = {r:.2f}; {slope*10:+.2f}% gold per +10 bp real yield")
print(f"\nNext Fed decision: {NEXT_MEETING:%B %d, %Y}")

In [ ]:
# Save results and download the charts (Colab only)
moves.round(3).to_csv(ROOT / "images" / "decision_day_moves.csv")
try:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/fed_results", "zip", IMAGES)
    files.download("/content/fed_results.zip")
except ImportError:
    print("Not in Colab: charts are saved in", IMAGES)